# Day 10 — Qwen3-VL-4B · Colab E0–E4 with checkpoint resume
Chọn T4 GPU rồi Run all để chạy lần lượt E0, E1, E2, E3 và E4 (rank 4/64). Code và dataset được tải tự động. E1–E4 cần Colab Secrets HF_TOKEN và HF_REPO_ID (repo private). Checkpoint đã đủ 300 step sẽ không train lại. Cuối lượt tải một ZIP tổng chứa kết quả của tất cả experiment.


## 0. Cấu hình & cài package (Colab T4)


In [1]:
EXPERIMENTS = ["E0", "E1", "E2", "E3", "E4"]  # Run in this order
E4_BASE = "E1"                     # Fixed baseline for the rank ablation
MICRO_BATCH_SIZE = 1                # Effective batch remains 8 with accumulation below
GRAD_ACCUM = 8
MAX_STEPS = 300
CHECKPOINT_EVERY = 25
EVAL_SPLIT = "test"                 # Use val only for debugging
REMOTE_BACKUP_REQUIRED = True
CODE_REF = "feat/day10-kaggle-qlora"
CODE_REVISION = "19525bd179bc2c3942dbd066fd1690c04e132572"  # Match the existing E1 checkpoint

DATA_SOURCE_DIR = ""                 # Optional extracted dataset directory
DATASET_ZIP = ""                     # Optional ZIP path or URL instead of public download

if not EXPERIMENTS or len(set(EXPERIMENTS)) != len(EXPERIMENTS):
    raise ValueError("Choose a non-empty list of unique experiments.")
if any(name not in ("E0", "E1", "E2", "E3", "E4") for name in EXPERIMENTS):
    raise ValueError("Experiments must be E0, E1, E2, E3, or E4.")


In [2]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content")
if not ROOT.is_dir():
    raise RuntimeError("Open this notebook in Google Colab before running it.")
REPO = ROOT / "K4-Track-4-Day10-VisionLanguageModel"
REPO_URL = "https://github.com/vbwork2/K4-Track-4-Day10-VisionLanguageModel.git"
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--branch", CODE_REF, REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_REVISION], cwd=REPO, check=True)
else:
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    if revision != CODE_REVISION:
        raise RuntimeError("The cached checkout has a different code revision. Use a fresh Colab runtime to preserve checkpoint compatibility.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "huggingface_hub", "kagglehub"], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "student_tests", "--basetemp", str(ROOT / "pytest-tmp")], cwd=REPO, check=True)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Repository:", REPO)
print("Code revision:", GIT_SHA)


Repository: /content/K4-Track-4-Day10-VisionLanguageModel
Code revision: 19525bd179bc2c3942dbd066fd1690c04e132572


## 1. GPU, Colab Secrets và dataset chuẩn của giảng viên
Dataset công khai được tải bằng kagglehub, rồi chia bởi script gốc. Nếu tải công khai bị chặn, đặt DATASET_ZIP hoặc DATA_SOURCE_DIR ở cell cấu hình. E1–E4 cần bật quyền Notebook access cho hai Hugging Face Secrets.


In [3]:
import json
import torch

from PIL import Image
from transformers import AutoProcessor
from student.checkpointing import require_remote_credentials
from student.data import prepare_public_dataset
from student.model import MODEL_ID
from scripts.labcommon import read_jsonl

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

HF_TOKEN = read_secret("HF_TOKEN")
HF_REPO_ID = read_secret("HF_REPO_ID")
if any(name != "E0" for name in EXPERIMENTS):
    require_remote_credentials(HF_TOKEN, HF_REPO_ID, required=REMOTE_BACKUP_REQUIRED)
DATA_DIR = ROOT / "day10_data"
RUN_ROOT = ROOT / "day10_runs"
DATASET = prepare_public_dataset(
    REPO, DATA_DIR, ROOT / "day10_prep",
    source_dir=DATA_SOURCE_DIR or None, zip_path=DATASET_ZIP or None,
)
ROWS = DATASET["rows"]
DATASET_VERSION = DATASET["meta"]["dataset_version"]
PROCESSOR = AutoProcessor.from_pretrained(MODEL_ID)
print("dataset_version:", DATASET_VERSION)
print("split counts:", DATASET["summary"])
print("Remote backup configured:", bool(HF_TOKEN and HF_REPO_ID))


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


100%|██████████| 185M/185M [00:00<00:00, 409MB/s]

Extracting files...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

dataset_version: d10-21e4070301
split counts: {'train': {'images': 1422, 'boxes': {'pothole': 1074, 'crack': 1750, 'manhole': 691}}, 'val': {'images': 276, 'boxes': {'pothole': 66, 'crack': 470, 'manhole': 103}}, 'test': {'images': 311, 'boxes': {'pothole': 121, 'crack': 299, 'manhole': 163}}}
Remote backup configured: True


## 2. Định dạng nhãn 0–1000, run manifest và fingerprint

Chuyển đổi nhãn, prompt và fingerprint được triển khai trong `student/`; đoạn này chạy kiểm tra dữ liệu và overlays.


In [4]:
import json

from student.labels import objects_to_answer, yolo_to_box
from student.visualization import render_ground_truth_overlays

sample = ROWS["train"][0]
print("Sample answer:", objects_to_answer(sample["objects"], sample["width"], sample["height"]))
print("One box:", yolo_to_box(sample["objects"][0], sample["width"], sample["height"]) if sample["objects"] else [])
LABEL_OVERLAYS = render_ground_truth_overlays(
    ROWS["train"], DATA_DIR, ROOT / "day10_label_overlays", limit=6
)
print("Six label overlays:", LABEL_OVERLAYS)


Sample answer: [{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_2d":[317,481,616,625]},{"label":"pothole","bbox_2d":[583,500,627,614]},{"label":"crack","bbox_2d":[580,508,655,614]},{"label":"pothole","bbox_2d":[259,517,311,572]},{"label":"crack","bbox_2d":[397,628,447,717]},{"label":"crack","bbox_2d":[339,683,398,808]},{"label":"crack","bbox_2d":[158,789,361,994]}]
One box: {'label': 'manhole', 'bbox_2d': [317, 481, 616, 625]}
Six label overlays: [PosixPath('/content/day10_label_overlays/ground_truth_01_20250216_164325.jpg'), PosixPath('/content/day10_label_overlays/ground_truth_02_20250216_164521.jpg'), PosixPath('/content/day10_label_overlays/ground_truth_03_20250216_164541.jpg'), PosixPath('/content/day10_label_overlays/ground_truth_04_20250219_164649.jpg'), PosixPath('/content/day10_label_overlays/ground_truth_05_20250219_164714.jpg'), PosixPath('/content/day10_label_overlays/ground_truth_06_20250219_164738.jpg'

## 3. Kiểm tra checkpoint trước khi load model

Runner tìm checkpoint tương thích mới nhất trong `/content` và private Hugging Face repo. Mismatch hoặc thiếu credential sẽ dừng trước khi train.


In [5]:
from student.checkpointing import require_remote_credentials
from student.experiments import run_id_for

if any(name != "E0" for name in EXPERIMENTS):
    require_remote_credentials(HF_TOKEN, HF_REPO_ID, required=REMOTE_BACKUP_REQUIRED)
PLANNED_RUN_IDS = []
for experiment in EXPERIMENTS:
    if experiment == "E4":
        PLANNED_RUN_IDS.extend(run_id_for("E4", E4_BASE, rank) for rank in (4, 64))
    else:
        PLANNED_RUN_IDS.append(run_id_for(experiment))
print("Run order:", PLANNED_RUN_IDS)
print("Compatible completed checkpoints skip training; evaluation is resumed locally or rerun on a fresh runtime.")


Run order: ['e0', 'e1', 'e2', 'e3', 'e4_rank_4_base_e1', 'e4_rank_64_base_e1']
Compatible completed checkpoints skip training; evaluation is resumed locally or rerun on a fresh runtime.


## 4. Qwen3-VL 4-bit: không lượng tử hoá ViT và `lm_head`; E1–E3 target module thực

`student/model.py` nạp NF4/double quant, xác nhận 252 target LLM và 96 target ViT, giữ nguyên `visual` và `lm_head`, rồi mở projector float32 cho E2/E3.


In [6]:
from student.model import EXPECTED_ARCHITECTURE_COUNTS
import student.model as model_helpers
from transformers import BitsAndBytesConfig

# Keep cached Colab checkouts compatible with full-path matching in Transformers 5.
def colab_quantization_config(torch_module):
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch_module.float16,
        bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["model.visual", "lm_head"],
    )
model_helpers.build_quantization_config = colab_quantization_config

# Install the verified overflow recovery loop into the cached checkout.
import importlib
import student.training as training_helpers
training_path = REPO / "student" / "training.py"
training_path.write_text('"""Resumable fp16 QLoRA training loop for E1, E2, and E3."""\n\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport subprocess\nimport sys\nimport time\n\nfrom student.checkpointing import (\n    atomic_write_json,\n    require_remote_credentials,\n    restore_checkpoint,\n    save_checkpoint,\n)\nfrom student.dataset import assert_image_token_count, collate_examples, encode_example\nfrom student.labels import objects_to_answer\n\n\n@dataclass(frozen=True)\nclass TrainingOptions:\n    setup: str\n    max_steps: int = 300\n    learning_rate: float = 2e-4\n    micro_batch_size: int = 1\n    grad_accum: int = 8\n    checkpoint_every: int = 25\n    seed: int = 42\n    max_grad_norm: float = 1.0\n    warmup_fraction: float = 0.05\n    coordinate_scale: str = "norm1000"\n    require_remote_backup: bool = True\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef git_revision(repo_root):\n    try:\n        return subprocess.check_output(\n            ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True, stderr=subprocess.DEVNULL\n        ).strip()\n    except Exception:\n        return "unknown"\n\n\ndef package_versions():\n    from importlib.metadata import PackageNotFoundError, version\n\n    output = {}\n    for package in ("torch", "transformers", "peft", "accelerate", "bitsandbytes", "trl", "huggingface_hub"):\n        try:\n            output[package] = version(package)\n        except PackageNotFoundError:\n            output[package] = "not-installed"\n    return output\n\n\ndef sample_indices(row_count, step, micro_step, batch_size, seed):\n    """Select deterministic training rows without depending on a dataloader iterator."""\n    if row_count <= 0 or batch_size <= 0:\n        raise ValueError("row_count and batch_size must be positive")\n    sampler = random.Random(seed + step * 8191 + micro_step)\n    return [sampler.randrange(row_count) for _ in range(batch_size)]\n\n\ndef last_logged_step(metrics_path):\n    path = Path(metrics_path)\n    if not path.is_file():\n        return 0\n    last = 0\n    with path.open(encoding="utf-8") as stream:\n        for line_number, line in enumerate(stream, 1):\n            if not line.strip():\n                continue\n            event = json.loads(line)\n            if event.get("event") != "train_step":\n                continue\n            step = int(event["step"])\n            if step <= last:\n                raise ValueError(f"Training metric steps must increase strictly; line {line_number} has {step}")\n            last = step\n    return last\n\n\ndef metric_summary(metrics_path):\n    events = []\n    path = Path(metrics_path)\n    if path.is_file():\n        with path.open(encoding="utf-8") as stream:\n            events = [json.loads(line) for line in stream if line.strip()]\n    steps = [event for event in events if event.get("event") == "train_step"]\n    return {\n        "global_step": int(steps[-1]["step"]) if steps else 0,\n        "peak_gpu_gb": max((float(event.get("peak_gpu_gb", 0)) for event in steps), default=0.0),\n        "elapsed_seconds": float(steps[-1].get("elapsed_seconds", 0)) if steps else 0.0,\n    }\n\n\ndef rebuild_train_log(metrics_path, train_log_path):\n    """Recreate the optional text log from the checkpointed canonical metrics."""\n    source = Path(metrics_path)\n    target = Path(train_log_path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="\\n") as output:\n        if source.is_file():\n            with source.open(encoding="utf-8") as events:\n                for line in events:\n                    event = json.loads(line)\n                    if event.get("event") != "train_step":\n                        continue\n                    output.write(\n                        f"step={event[\'step\']} loss={event[\'loss\']:.6f} lr={event[\'learning_rate\']:.9g} "\n                        f"grad_norm={event[\'grad_norm\']:.5f} peak_gpu_gb={event[\'peak_gpu_gb\']:.3f}\\n"\n                    )\n        output.flush()\n        os.fsync(output.fileno())\n    os.replace(temporary, target)\n\n\ndef append_metric(metrics_path, event, expected_previous_step):\n    path = Path(metrics_path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    step = int(event["step"])\n    if step != expected_previous_step + 1:\n        raise ValueError(f"Refusing non-monotonic metric step {step} after {expected_previous_step}")\n    with path.open("a", encoding="utf-8", newline="\\n") as stream:\n        stream.write(json.dumps(event, ensure_ascii=False, allow_nan=False) + "\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n\n\ndef _write_train_log(path, event):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8", newline="\\n") as stream:\n        stream.write(\n            f"step={event[\'step\']} loss={event[\'loss\']:.6f} lr={event[\'learning_rate\']:.9g} "\n            f"grad_norm={event[\'grad_norm\']:.5f} peak_gpu_gb={event[\'peak_gpu_gb\']:.3f}\\n"\n        )\n        stream.flush()\n        os.fsync(stream.fileno())\n\n\ndef _encode_record(record, data_dir, processor, coordinate_scale):\n    from PIL import Image\n\n    with Image.open(Path(data_dir) / record["image_path"]) as source:\n        image = source.convert("RGB")\n    answer = objects_to_answer(\n        record["objects"],\n        width=record["width"],\n        height=record["height"],\n        coord=coordinate_scale,\n    )\n    return encode_example(processor, image, answer)\n\n\ndef _autocast(torch):\n    try:\n        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)\n    except AttributeError:\n        return torch.cuda.amp.autocast(dtype=torch.float16)\n\n\ndef _grad_scaler(torch):\n    try:\n        return torch.amp.GradScaler("cuda", enabled=True, init_scale=1024.0)\n    except AttributeError:\n        return torch.cuda.amp.GradScaler(enabled=True, init_scale=1024.0)\n\n\ndef _try_optimizer_step(torch, optimizer, scaler, train_params, max_grad_norm):\n    """Reduce loss scale after overflow without counting an optimizer update."""\n    scaler.unscale_(optimizer)\n    # Check individual gradients before clipping, which can spread NaNs.\n    checks = [torch.isfinite(parameter.grad).all() for parameter in train_params if parameter.grad is not None]\n    if not checks:\n        raise RuntimeError("No trainable parameter received a gradient")\n    finite = torch.stack(checks).all().item()\n    if not finite:\n        scaler.update()\n        optimizer.zero_grad(set_to_none=True)\n        return None\n    grad_norm = float(torch.nn.utils.clip_grad_norm_(train_params, max_grad_norm).item())\n    if not math.isfinite(grad_norm):\n        optimizer.zero_grad(set_to_none=True)\n        raise FloatingPointError("Gradient norm overflowed even though individual gradients are finite")\n    learning_rate = float(optimizer.param_groups[0]["lr"])\n    scaler.step(optimizer)\n    scaler.update()\n    optimizer.zero_grad(set_to_none=True)\n    return grad_norm, learning_rate\n\n\ndef _run_manifest(repo_root, dataset_version, options, config_fingerprint, trainable_params,\n                  run_id, run_metadata=None):\n    manifest = {\n        "run_id": run_id,\n        "git_sha": git_revision(repo_root),\n        "dataset_version": dataset_version,\n        "setup": options.setup,\n        "config_fingerprint": config_fingerprint,\n        "model_id": "Qwen/Qwen3-VL-4B-Instruct",\n        "seed": options.seed,\n        "training_options": asdict(options),\n        "trainable_params": int(trainable_params),\n        "package_versions": package_versions(),\n        "started_at_utc": utc_now(),\n        "status": "initialized",\n    }\n    if run_metadata:\n        manifest["config"] = run_metadata\n    return manifest\n\n\ndef train_model(\n    bundle,\n    train_rows,\n    data_dir,\n    repo_root,\n    run_dir,\n    options,\n    config_fingerprint,\n    dataset_version,\n    remote_repo_id=None,\n    token=None,\n    resume_checkpoint=None,\n    run_metadata=None,\n    on_step=None,\n):\n    """Train one supervised setup and persist a verified checkpoint every configured interval."""\n    import numpy as np\n    import torch\n    from transformers import get_cosine_schedule_with_warmup\n\n    if options.setup not in ("E1", "E2", "E3"):\n        raise ValueError("train_model accepts E1, E2, or E3")\n    if options.max_steps <= 0 or options.grad_accum <= 0 or options.micro_batch_size <= 0:\n        raise ValueError("Training steps and batch sizes must be positive")\n    if options.checkpoint_every <= 0:\n        raise ValueError("checkpoint_every must be positive")\n    require_remote_credentials(token, remote_repo_id, required=options.require_remote_backup)\n    if not torch.cuda.is_available():\n        raise RuntimeError("QLoRA training requires a CUDA GPU")\n    if not train_rows:\n        raise ValueError("Training split is empty")\n    if bundle.setup != options.setup:\n        raise ValueError(f"Loaded model setup {bundle.setup} does not match {options.setup}")\n\n    run_dir = Path(run_dir).resolve()\n    run_dir.mkdir(parents=True, exist_ok=True)\n    run_id = run_dir.name\n    metrics_path = run_dir / "metrics.jsonl"\n    train_log_path = run_dir / "train.log"\n    train_params = [parameter for parameter in bundle.model.parameters() if parameter.requires_grad]\n    if not train_params or sum(parameter.numel() for parameter in train_params) != bundle.trainable_params:\n        raise RuntimeError("Loaded model trainable parameter list changed before training")\n\n    random.seed(options.seed)\n    np.random.seed(options.seed)\n    torch.manual_seed(options.seed)\n    torch.cuda.manual_seed_all(options.seed)\n    optimizer = torch.optim.AdamW(train_params, lr=options.learning_rate, weight_decay=0.0)\n    warmup_steps = max(1, round(options.max_steps * options.warmup_fraction))\n    scheduler = get_cosine_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=warmup_steps,\n        num_training_steps=options.max_steps,\n    )\n    scaler = _grad_scaler(torch)\n    manifest = _run_manifest(\n        repo_root,\n        dataset_version,\n        options,\n        config_fingerprint,\n        bundle.trainable_params,\n        run_id,\n        run_metadata=run_metadata,\n    )\n    active_run_json = run_dir / "run.json"\n    atomic_write_json(active_run_json, manifest)\n\n    starting_step = 0\n    elapsed_before = 0.0\n    if resume_checkpoint is not None:\n        restored = restore_checkpoint(\n            resume_checkpoint,\n            optimizer,\n            scheduler,\n            scaler,\n            projector_modules=bundle.projector_modules,\n            active_metrics_path=metrics_path,\n            config_fingerprint=config_fingerprint,\n            dataset_version=dataset_version,\n            setup=options.setup,\n        )\n        starting_step = restored["global_step"]\n        elapsed_before = restored["elapsed_seconds"]\n        rebuild_train_log(metrics_path, train_log_path)\n        if starting_step >= options.max_steps:\n            summary = metric_summary(metrics_path)\n            manifest.update({\n                "status": "trained",\n                "completed_steps": starting_step,\n                "train_minutes": max(elapsed_before, summary["elapsed_seconds"]) / 60,\n                "peak_gpu_gb": summary["peak_gpu_gb"],\n                "finished_at_utc": utc_now(),\n            })\n            atomic_write_json(active_run_json, manifest)\n            return {"status": "already_complete", "global_step": starting_step, "run_manifest": manifest}\n    else:\n        metrics_path.write_text("", encoding="utf-8")\n        initial_manifest = dict(manifest, status="training", start_step=0)\n        save_checkpoint(\n            run_dir=run_dir,\n            global_step=0,\n            model=bundle.model,\n            optimizer=optimizer,\n            scheduler=scheduler,\n            scaler=scaler,\n            setup=options.setup,\n            config_fingerprint=config_fingerprint,\n            dataset_version=dataset_version,\n            trainable_params=bundle.trainable_params,\n            elapsed_seconds=0.0,\n            metrics_path=metrics_path,\n            run_manifest=initial_manifest,\n            projector_modules=bundle.projector_modules,\n            remote_repo_id=remote_repo_id,\n            token=token,\n            remote_required=options.require_remote_backup,\n            checkpoint_every=options.checkpoint_every,\n        )\n\n    previous_step = last_logged_step(metrics_path)\n    if previous_step != starting_step:\n        raise RuntimeError(\n            f"Checkpoint step {starting_step} does not match metrics log step {previous_step}; refusing duplicate metrics"\n        )\n    probe = _encode_record(train_rows[0], data_dir, bundle.processor, options.coordinate_scale)\n    assert_image_token_count(bundle.processor, probe, expected=220)\n    del probe\n\n    bundle.model.train()\n    optimizer.zero_grad(set_to_none=True)\n    torch.cuda.reset_peak_memory_stats()\n    started = time.monotonic()\n    manifest.update({"status": "training", "start_step": starting_step, "resumed_from": str(resume_checkpoint) if resume_checkpoint else None})\n    atomic_write_json(active_run_json, manifest)\n    try:\n        for step in range(starting_step + 1, options.max_steps + 1):\n            for attempt in range(12):\n                loss_total = 0.0\n                for micro_step in range(options.grad_accum):\n                    indices = sample_indices(\n                        len(train_rows), step, micro_step, options.micro_batch_size, options.seed\n                    )\n                    examples = [\n                        _encode_record(train_rows[index], data_dir, bundle.processor, options.coordinate_scale)\n                        for index in indices\n                    ]\n                    batch = collate_examples(examples, bundle.processor, device="cuda:0")\n                    with _autocast(torch):\n                        output = bundle.model(**batch)\n                        loss = output.loss / options.grad_accum\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError(f"Non-finite loss at optimizer step {step}: {loss.detach().item()}")\n                    scaler.scale(loss).backward()\n                    loss_total += float(loss.detach().item()) * options.grad_accum\n                    del batch, examples, output, loss\n\n                update = _try_optimizer_step(torch, optimizer, scaler, train_params, options.max_grad_norm)\n                if update is not None:\n                    grad_norm, learning_rate_used = update\n                    break\n                print(f"Retry step {step} after FP16 gradient overflow; loss scale={scaler.get_scale():g}", flush=True)\n            else:\n                raise FloatingPointError(f"Persistent non-finite gradients at optimizer step {step} after 12 attempts")\n            scheduler.step()\n\n            elapsed = elapsed_before + time.monotonic() - started\n            duration = max(elapsed - float(manifest.get("last_elapsed_seconds", elapsed_before)), 1e-9)\n            event = {\n                "event": "train_step",\n                "run_id": run_id,\n                "setup": options.setup,\n                "step": step,\n                "loss": loss_total / options.grad_accum,\n                "learning_rate": learning_rate_used,\n                "grad_norm": grad_norm,\n                "grad_scale": float(scaler.get_scale()),\n                "elapsed_seconds": elapsed,\n                "peak_gpu_gb": float(torch.cuda.max_memory_allocated() / (1024 ** 3)),\n                "images_per_second": float(options.micro_batch_size * options.grad_accum / duration),\n                "timestamp_utc": utc_now(),\n            }\n            append_metric(metrics_path, event, previous_step)\n            _write_train_log(train_log_path, event)\n            previous_step = step\n            manifest["last_elapsed_seconds"] = elapsed\n            if on_step is not None:\n                on_step(event)\n            if step % options.checkpoint_every == 0 or step == options.max_steps:\n                manifest.update({"status": "training", "last_checkpoint_step": step})\n                atomic_write_json(active_run_json, manifest)\n                save_checkpoint(\n                    run_dir=run_dir,\n                    global_step=step,\n                    model=bundle.model,\n                    optimizer=optimizer,\n                    scheduler=scheduler,\n                    scaler=scaler,\n                    setup=options.setup,\n                    config_fingerprint=config_fingerprint,\n                    dataset_version=dataset_version,\n                    trainable_params=bundle.trainable_params,\n                    elapsed_seconds=elapsed,\n                    metrics_path=metrics_path,\n                    run_manifest=manifest,\n                    projector_modules=bundle.projector_modules,\n                    remote_repo_id=remote_repo_id,\n                    token=token,\n                    remote_required=options.require_remote_backup,\n                    checkpoint_every=options.checkpoint_every,\n                )\n\n        elapsed_total = elapsed_before + time.monotonic() - started\n        summary = metric_summary(metrics_path)\n        manifest.update({\n            "status": "trained",\n            "completed_steps": options.max_steps,\n            "train_minutes": elapsed_total / 60,\n            "peak_gpu_gb": max(summary["peak_gpu_gb"], float(torch.cuda.max_memory_allocated() / (1024 ** 3))),\n            "finished_at_utc": utc_now(),\n        })\n        atomic_write_json(active_run_json, manifest)\n        return {"status": "trained", "global_step": options.max_steps, "run_manifest": manifest}\n    except Exception as error:\n        error_text = str(error)\n        if token:\n            error_text = error_text.replace(token, "[REDACTED]")\n        manifest.update({\n            "status": "interrupted",\n            "error_type": type(error).__name__,\n            "error_message": error_text[:1000],\n            "last_checkpoint_step": max(\n                [0] + [int(path.name.removeprefix("step_")) for path in (run_dir / "checkpoints").glob("step_*") if path.is_dir() and not path.name.endswith(".tmp")]\n            ),\n            "finished_at_utc": utc_now(),\n        })\n        atomic_write_json(active_run_json, manifest)\n        raise\n\n', encoding="utf-8")
importlib.reload(training_helpers)
import student.experiments as experiment_helpers
experiment_helpers.train_model = training_helpers.train_model
print("Quantization exclusions and FP16 overflow recovery installed.")


Quantization exclusions and FP16 overflow recovery installed.


## 5. Encode ví dụ, mask assistant-only loss; batch vision concat

Notebook chạy prompt-prefix, assistant-label mask, 220 image token và collation qua processor thật trước khi train.


In [7]:
from PIL import Image

from student.dataset import assert_image_token_count, collate_examples, encode_example
from student.labels import objects_to_answer

sample = ROWS["train"][0]
with Image.open(DATA_DIR / sample["image_path"]) as source:
    image = source.convert("RGB")
answer = objects_to_answer(sample["objects"], sample["width"], sample["height"])
encoded = encode_example(PROCESSOR, image, answer)
image_token_count = assert_image_token_count(PROCESSOR, encoded, expected=220)
print("Image tokens:", image_token_count)
print("Assistant-only answer:", PROCESSOR.tokenizer.decode(encoded["labels"][encoded["labels"] != -100], skip_special_tokens=False))
print("Collated fields:", {key: tuple(value.shape) for key, value in collate_examples([encoded], PROCESSOR).items()})


Image tokens: 220
Assistant-only answer: [{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_2d":[317,481,616,625]},{"label":"pothole","bbox_2d":[583,500,627,614]},{"label":"crack","bbox_2d":[580,508,655,614]},{"label":"pothole","bbox_2d":[259,517,311,572]},{"label":"crack","bbox_2d":[397,628,447,717]},{"label":"crack","bbox_2d":[339,683,398,808]},{"label":"crack","bbox_2d":[158,789,361,994]}]<|im_end|>

Collated fields: {'image_grid_thw': (1, 3), 'pixel_values': (880, 1536), 'attention_mask': (1, 579), 'labels': (1, 579), 'input_ids': (1, 579), 'mm_token_type_ids': (1, 579)}


## 6. Save + restore full training state: adapter, projector, optimizer state và logs

Checkpoint lưu theo step, có integrity manifest; token HF chỉ được đọc từ Colab Secrets và không được ghi vào run manifest.


In [8]:
from student.checkpointing import local_checkpoints

for run_id in PLANNED_RUN_IDS:
    print(run_id, "local checkpoint directories:", len(local_checkpoints(RUN_ROOT / run_id)))
print("Remote backup is fail-closed:", REMOTE_BACKUP_REQUIRED)
print("Remote checkpoint fingerprints are checked before loading each model.")


e0 local checkpoint directories: 0
e1 local checkpoint directories: 0
e2 local checkpoint directories: 0
e3 local checkpoint directories: 0
e4_rank_4_base_e1 local checkpoint directories: 0
e4_rank_64_base_e1 local checkpoint directories: 0
Remote backup is fail-closed: True
Remote checkpoint fingerprints are checked before loading each model.


## 7. Chạy toàn bộ E0–E4 theo thứ tự

Chỉ cần Run all. E4 tự chạy rank 4 và 64 với baseline E1. Mỗi experiment được đóng ZIP ngay sau khi xong; model được giải phóng trước experiment tiếp theo. Khi bị ngắt, chạy lại cùng notebook/cấu hình/Secrets để khôi phục checkpoint; E1 đã đủ 300 step sẽ không train lại. Các file chỉ nằm trong /content có thể mất khi runtime bị xóa, nên tải ZIP đã hoàn thành nếu lượt chạy dừng giữa chừng.


In [10]:
%pip uninstall -y torchao

import importlib
import sys
import gc
import torch
from peft.import_utils import is_torchao_available

importlib.invalidate_caches()
is_torchao_available.cache_clear()

for name in ("last_value", "last_traceback", "last_exc"):
    setattr(sys, name, None)
gc.collect()
torch.cuda.empty_cache()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [11]:
import gc
import json
import zipfile

from student.experiments import compare_rank_evidence, run_experiment, run_rank_ablation

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU disappeared before the experiments started.")

def report_step(event):
    if event["step"] == 1 or event["step"] % CHECKPOINT_EVERY == 0:
        print(event["run_id"], "STEP", event["step"], "loss", round(event["loss"], 5), "peak GB", round(event["peak_gpu_gb"], 2), flush=True)

def package_results(experiment, results):
    artifact = ROOT / f"{experiment.lower()}_logs_evidence.zip"
    with zipfile.ZipFile(artifact, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for result in results:
            run_id = result["run_id"]
            run_dir = RUN_ROOT / run_id
            for path in sorted(run_dir.rglob("*")):
                relative = path.relative_to(run_dir)
                if path.is_file() and "checkpoints" not in relative.parts and path.suffix in (".json", ".jsonl", ".log"):
                    archive.write(path, arcname=f"runs/{run_id}/{relative.as_posix()}")
            evidence_path = Path(result["evidence"]["evidence_path"])
            archive.write(evidence_path, arcname=f"evidence/{run_id}.json")
            for figure in result["evidence"].get("annotated_examples", []):
                path = Path(figure)
                archive.write(path, arcname=f"evidence/figures/{run_id}/{path.name}")
    return artifact

ALL_RESULTS = []
ARTIFACTS = []
for experiment in EXPERIMENTS:
    print("Starting", experiment, flush=True)
    current_results = []
    try:
        controls = dict(
            micro_batch_size=MICRO_BATCH_SIZE, grad_accum=GRAD_ACCUM,
            max_steps=MAX_STEPS, checkpoint_every=CHECKPOINT_EVERY,
            remote_repo_id=HF_REPO_ID, token=HF_TOKEN,
            remote_required=REMOTE_BACKUP_REQUIRED, on_step=report_step,
        )
        if experiment == "E4":
            current_results = run_rank_ablation(
                REPO, DATA_DIR, RUN_ROOT, base_setup=E4_BASE, split=EVAL_SPLIT, **controls,
            )
            if EVAL_SPLIT == "test":
                E4_COMPARISON = compare_rank_evidence(
                    current_results[0]["evidence"]["evidence_path"],
                    current_results[1]["evidence"]["evidence_path"],
                )
                print(json.dumps(E4_COMPARISON, ensure_ascii=False, indent=2))
        else:
            current_results.append(run_experiment(
                experiment, REPO, DATA_DIR, RUN_ROOT, split=EVAL_SPLIT,
                evaluate=True, **controls,
            ))
        # Release GPU references before starting the next experiment.
        for result in current_results:
            result.pop("bundle", None)
        ARTIFACTS.append(package_results(experiment, current_results))
        ALL_RESULTS.extend(current_results)
        print("Completed", experiment, "archive:", ARTIFACTS[-1], flush=True)
    finally:
        for result in current_results:
            result.pop("bundle", None)
        gc.collect()
        torch.cuda.empty_cache()


Starting E0


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Completed E0 archive: /content/e0_logs_evidence.zip
Starting E1


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

...

e3 STEP 1 loss 1.12749 peak GB 9.12
e3 STEP 25 loss 0.70441 peak GB 9.92
e3 STEP 50 loss 0.54177 peak GB 9.98
e3 STEP 75 loss 0.50927 peak GB 9.98
e3 STEP 100 loss 0.51508 peak GB 10.18
e3 STEP 125 loss 0.53491 peak GB 10.18
e3 STEP 150 loss 0.48919 peak GB 10.18
e3 STEP 175 loss 0.4682 peak GB 10.18
e3 STEP 200 loss 0.54194 peak GB 10.18
e3 STEP 225 loss 0.49373 peak GB 10.18
e3 STEP 250 loss 0.43023 peak GB 10.18
e3 STEP 275 loss 0.38552 peak GB 10.18
e3 STEP 300 loss 0.51233 peak GB 10.18
Completed E3 archive: /content/e3_logs_evidence.zip
Starting E4


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

e4_rank_4_base_e1 STEP 1 loss 1.12651 peak GB 7.59
e4_rank_4_base_e1 STEP 25 loss 0.77875 peak GB 7.94
e4_rank_4_base_e1 STEP 50 loss 0.56183 peak GB 8.0
e4_rank_4_base_e1 STEP 75 loss 0.59836 peak GB 8.04
e4_rank_4_base_e1 STEP 100 loss 0.53438 peak GB 8.2
e4_rank_4_base_e1 STEP 125 loss 0.6181 peak GB 8.2
e4_rank_4_base_e1 STEP 150 loss 0.56333 peak GB 8.2
e4_rank_4_base_e1 STEP 175 loss 0.54947 peak GB 8.2
e4_rank_4_base_e1 STEP 200 loss 0.64259 peak GB 8.2
e4_rank_4_base_e1 STEP 225 loss 0.56176 peak GB 8.2
e4_rank_4_base_e1 STEP 250 loss 0.55171 peak GB 8.2
e4_rank_4_base_e1 STEP 275 loss 0.50427 peak GB 8.2
e4_rank_4_base_e1 STEP 300 loss 0.55382 peak GB 8.2


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

e4_rank_64_base_e1 STEP 1 loss 1.12651 peak GB 9.05
e4_rank_64_base_e1 STEP 25 loss 0.68581 peak GB 9.81
e4_rank_64_base_e1 STEP 50 loss 0.53575 peak GB 9.88
e4_rank_64_base_e1 STEP 75 loss 0.57894 peak GB 9.88
e4_rank_64_base_e1 STEP 100 loss 0.55531 peak GB 10.07
e4_rank_64_base_e1 STEP 125 loss 0.61135 peak GB 10.07
e4_rank_64_base_e1 STEP 150 loss 0.57112 peak GB 10.07
e4_rank_64_base_e1 STEP 175 loss 0.54773 peak GB 10.07
e4_rank_64_base_e1 STEP 200 loss 0.62157 peak GB 10.07
e4_rank_64_base_e1 STEP 225 loss 0.52906 peak GB 10.07
e4_rank_64_base_e1 STEP 250 loss 0.52355 peak GB 10.07
e4_rank_64_base_e1 STEP 275 loss 0.4758 peak GB 10.07
e4_rank_64_base_e1 STEP 300 loss 0.52893 peak GB 10.07
{
  "dataset_version": "d10-21e4070301",
  "split": "test",
  "rank_4": {
    "f1": 0.2670299727520436,
    "ci95": [
      0.2325189553496209,
      0.3068391866913124
    ],
    "trainable_params": 8257536,
    "peak_vram_gb": 8.197035789489746,
    "train_minutes": 10.573612314233333
  },
  

## 8. Suy luận, score chính thức và evidence

Suy luận JSONL được flush theo từng ảnh. Chỉ tập test đầy đủ mới được chuyển cho `scripts/score.py`; evidence và overlay chỉ được tạo từ dự đoán thật.


In [12]:
for result in ALL_RESULTS:
    evidence = result["evidence"]
    print(result["run_id"], "F1:", evidence["micro"]["f1"],
          "images:", evidence["n_images"], "missing:", evidence["missing_predictions"])
    print("Predictions:", result["predictions"])
print("Completed runs:", [result["run_id"] for result in ALL_RESULTS])


e0 F1: 0.09690721649484538 images: 311 missing: 0
Predictions: /content/day10_runs/e0/predictions_test.jsonl
e1 F1: 0.2631094756209751 images: 311 missing: 0
Predictions: /content/day10_runs/e1/predictions_test.jsonl
e2 F1: 0.33641404805914965 images: 311 missing: 0
Predictions: /content/day10_runs/e2/predictions_test.jsonl
e3 F1: 0.42435775451950525 images: 311 missing: 0
Predictions: /content/day10_runs/e3/predictions_test.jsonl
e4_rank_4_base_e1 F1: 0.2670299727520436 images: 311 missing: 0
Predictions: /content/day10_runs/e4_rank_4_base_e1/predictions_test.jsonl
e4_rank_64_base_e1 F1: 0.266173752310536 images: 311 missing: 0
Predictions: /content/day10_runs/e4_rank_64_base_e1/predictions_test.jsonl
Completed runs: ['e0', 'e1', 'e2', 'e3', 'e4_rank_4_base_e1', 'e4_rank_64_base_e1']


## 9. Tải kết quả của toàn bộ lượt chạy

ZIP tổng có logs, predictions, evidence và ảnh minh họa của từng run, cùng ZIP riêng của mỗi experiment. Không chứa trọng số hoặc checkpoint. Nếu bị ngắt, ZIP của experiment đã xong nằm ở /content; tải các file đó trước khi đổi runtime. Checkpoint training lưu trên private HF repo.


In [13]:
import zipfile

completed_run_ids = [result["run_id"] for result in ALL_RESULTS]
if completed_run_ids != PLANNED_RUN_IDS:
    raise RuntimeError("Some experiments are incomplete; download completed ZIP files from /content before restarting.")
ARTIFACT = ROOT / "e0_e4_logs_evidence.zip"
with zipfile.ZipFile(ARTIFACT, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in ARTIFACTS:
        archive.write(artifact, arcname=f"archives/{artifact.name}")
        with zipfile.ZipFile(artifact) as individual:
            for name in individual.namelist():
                archive.writestr(name, individual.read(name))
print("Downloadable logs and evidence:", ARTIFACT)
print("Completed experiments:", EXPERIMENTS)

from google.colab import files
files.download(str(ARTIFACT))


Downloadable logs and evidence: /content/e0_e4_logs_evidence.zip
Completed experiments: ['E0', 'E1', 'E2', 'E3', 'E4']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Checklist trước khi nộp

- GPU T4 đã bật và dataset được attach.
- E1–E4 có private HF repo cùng Secrets hợp lệ.
- E0–E3 và cả rank 4/64 của E4 đã chạy thật trước khi điền `REPORT.md`.
- Đã kiểm tra `dataset_version`, `missing_predictions`, metadata VRAM/thời gian và 6 ảnh overlays.
- GRPO tùy chọn được ghi `NOT_RUN` nếu chưa thực hiện.
